In [4]:
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

from utils import write_csv
from utils import display_excel_sheets

from models import (
    AssessmentReport,
    DataSource,
    GasType,
    Emissions,
    EmissionsCO2e,
    EmissionsTotalCO2e,
    EmissionsTotalSector,
    AggregationType,
)



In [5]:
datasource_name = "c40-cities"
fl = (
    Path.cwd().resolve().parent.parent/ "data" / f"{datasource_name}" / "raw" / "C40_GPC_Database.xlsx"
)

In [6]:
display_excel_sheets(fl)

['Cover', 'GHG Dashboard Data - Inventory', 'GPC taxonomy']

In [7]:

sheet_name =  'GHG Dashboard Data - Inventory'
df_raw = pd.read_excel(fl, sheet_name=sheet_name)

df_gpc = pd.read_excel(fl, sheet_name= 'GPC taxonomy')

In [8]:
#list(df_raw.columns)
columns = [
 'City',
 'Country',
 'Year_calendar',
 'Population',
 'Area',
 'GDP',
 'I.1.1',
 'I.1.2',
 'I.1.3',
 'I.2.1',
 'I.2.2',
 'I.2.3',
 'I.3.1',
 'I.3.2',
 'I.3.3',
 'I.4.1',
 'I.4.2',
 'I.4.3',
 'I.4.4',
 'I.5.1',
 'I.5.2',
 'I.5.3',
 'I.6.1',
 'I.6.2',
 'I.6.3',
 'I.7.1',
 'I.8.1',
 'II.1.1',
 'II.1.2',
 'II.1.3',
 'II.2.1',
 'II.2.2',
 'II.2.3',
 'II.3.1',
 'II.3.2',
 'II.3.3',
 'II.4.1',
 'II.4.2',
 'II.4.3',
 'II.5.1',
 'II.5.2',
 'II.5.3',
 'III.1.1',
 'III.1.2',
 'III.1.3',
 'III.2.1',
 'III.2.2',
 'III.2.3',
 'III.3.1',
 'III.3.2',
 'III.3.3',
 'III.4.1',
 'III.4.2',
 'III.4.3',
 'IV.1',
 'IV.2',
 'V.1',
 'V.2',
 'V.3',
 'VI.1',
]

id_vars = [ 
    'City',
    'Country',
    'Year_calendar',
    'Population',
    'Area',
    'GDP'
    ]

subsector_codes = [c for c in columns if c not in id_vars]

rename_dictionary = {
    "Côte d’Ivoire":"Côte d'Ivoire",
    "The Netherlands":"Netherlands",
    "USA":"United States",
    "Colombia ":"Colombia",
    "Vietnam":"Viet Nam",
    "Turkey":"Türkiye"
}

df_emissions = (
    df_raw
    .loc[:, columns]
    .melt(
            id_vars=id_vars, value_vars=subsector_codes, var_name="sector", value_name="emissions"
        )
    #.query("emissions not in ['NE', 'IE', 'NO', 'C', 'IE (I.1.1)', 'IE (I.1.1)']")
    .assign(emissions=lambda x: pd.to_numeric(x["emissions"], errors="coerce"))
    .dropna(subset=["emissions"])
    .assign(Country=lambda d: d["Country"].replace(rename_dictionary))
)


In [9]:
# ------------------------------------------
# get actors
# ------------------------------------------

fl_actors = (
    Path().cwd().parent.parent
    #Path(__file__).resolve().parent.parent.parent
    / "data"
    / "iso-3166-1"
    / "raw"
    / "countries.csv"
)

df_actors = pd.read_csv(fl_actors, keep_default_na=False)[
    ["#country_code_alpha2", "country_code_alpha3", "name_short"]
].rename(
    columns={"#country_code_alpha2": "actor_id", "country_code_alpha3": "iso3", "name_short":"name"}
)

In [10]:


df_country = (
    df_raw['Country'].drop_duplicates().to_frame()
    .assign(Country=lambda d: d["Country"].replace(rename_dictionary))
    .rename(columns={"Country":"name"})
    .merge(df_actors, on="name", how="left")
)

In [11]:
#df_raw.assign(Country=lambda d: d["Country"].replace(rename_dictionary))

In [12]:
locode_file = (
    Path().cwd().parent.parent
    #Path(__file__).resolve().parent.parent.parent
    / "data"
    / "locode"
    / "processed"
    / "actor.csv"
)

df_locode = (
    pd.read_csv(locode_file)
    .loc[:,["id", "is_part_of", "name"]]
    .rename(columns={"id":"locode", "name":"city", "is_part_of":"iso"})
    #.loc[lambda df: df["iso"].str.contains("US-", na=False)]
)



In [13]:
df_locode

,locode,iso,city
0,ADALV,AD,Andorra la Vella
1,ADCAN,AD,Canillo
2,ADENC,AD,Encamp
3,ADESC,AD,Escaldes-Engordany
4,ADEAC,AD-04,Escas
...,...,...,...
112606,ZWSTH,ZW,Southerton
112607,ZWTHJ,ZW,Thompson Junction
112608,ZWVFA,ZW,Victoria Falls
112609,ZWZMZ,ZW-MV,Zimbabwe


In [14]:
df_cities = df_emissions[["City", "Country"]].drop_duplicates().rename(columns={"City":"city", "Country":"name"}).merge(df_country, on="name").rename(columns={"actor_id":"iso"})

In [15]:
df_locode

,locode,iso,city
0,ADALV,AD,Andorra la Vella
1,ADCAN,AD,Canillo
2,ADENC,AD,Encamp
3,ADESC,AD,Escaldes-Engordany
4,ADEAC,AD-04,Escas
...,...,...,...
112606,ZWSTH,ZW,Southerton
112607,ZWTHJ,ZW,Thompson Junction
112608,ZWVFA,ZW,Victoria Falls
112609,ZWZMZ,ZW-MV,Zimbabwe


In [16]:
list(df_cities["city"])

['Abidjan',
 'Accra',
 'Addis Ababa',
 'Ahmedabad',
 'Amman',
 'Amsterdam',
 'Amsterdam',
 'Athens',
 'Auckland',
 'Austin',
 'Bangkok',
 'Barcelona',
 'Bengaluru',
 'Berlin',
 'Bogotá',
 'Boston',
 'Buenos Aires',
 'Cape Town',
 'Chennai',
 'Chicago',
 'Copenhagen',
 'Curitiba',
 'Dakar',
 'Delhi NCT',
 'Dhaka North',
 'Dhaka South',
 'Dubai',
 'Durban (eThekwini)',
 'Ekurhuleni',
 'Guadalajara',
 'Hanoi',
 'Heidelberg',
 'Houston',
 'Istanbul',
 'Johannesburg',
 'Kolkata',
 'Kuala Lumpur',
 'Lagos',
 'Lima',
 'Lisbon',
 'London',
 'Los Angeles',
 'Madrid',
 'Medellín',
 'Melbourne',
 'Ciudad de México',
 'Miami',
 'Milan',
 'Montréal',
 'Mumbai',
 'Nairobi',
 'New Orleans',
 'New York City',
 'Oslo',
 'Paris',
 'Philadelphia',
 'Phoenix',
 'Portland',
 'Quezon City',
 'Quito',
 'Rio de Janeiro',
 'Rome',
 'Rotterdam',
 'Rotterdam',
 'Salvador',
 'San Francisco',
 'Santiago',
 'São Paulo',
 'Seattle',
 'Seoul',
 'Stockholm',
 'Sydney',
 'Tel Aviv',
 'Tokyo',
 'Toronto',
 'Tshwane',
 '

In [17]:
df_locode.assign(iso = lambda d: d["iso"].apply(lambda x: x[:2]))

,locode,iso,city
0,ADALV,AD,Andorra la Vella
1,ADCAN,AD,Canillo
2,ADENC,AD,Encamp
3,ADESC,AD,Escaldes-Engordany
4,ADEAC,AD,Escas
...,...,...,...
112606,ZWSTH,ZW,Southerton
112607,ZWTHJ,ZW,Thompson Junction
112608,ZWVFA,ZW,Victoria Falls
112609,ZWZMZ,ZW,Zimbabwe


In [18]:
city_rename = {
  "Athens": "Athinai", 
  "Bengaluru":"Bangalore",
  "Bogotá":"Bogota",
  "Chennai":"Chennai (ex Madras)",
  "Copenhagen": "Kobenhavn",
  'Delhi NCT': "Delhi",
  'Durban (eThekwini)': "Durban",
  'Kolkata': "Kolkata (ex Calcutta)",
  'Lisbon': "Lisboa",
  'Medellín': "Medellin",
  'Ciudad de México': "Ciudad de Mexico",
  'Milan': "Milano",
  'Montréal': "Montreal",
  'Mumbai': "Mumbai (ex Bombay)",
  'Rome': "Rome (Roma)",
  'Tel Aviv': "Tel Aviv-Yafo",
  'Tshwane': "Pretoria",
  'Warsaw': "Warszawa",

  # deal this this special case
  # maybe make a new locode for these?
  # BD DAC 
  #'Dhaka North': "", BD DAN ?
  #'Dhaka South': "", BD DAS ?

  #'Ekurhuleni': "", # no single locodoe for this city

  # set these by hand
  #'New York City': "New York", # need to make sure this puts USNYC for locode and not one from TX or MN
  #'São Paulo': "Sao Paulo", #make sure gets BRSAO
  #'Washington, DC': "Washington"  # make sure is USWAS
}

In [19]:
df_merged = (
    df_cities.assign(city=lambda d: d["city"].replace(city_rename))
    .merge(df_locode.assign(iso = lambda d: d["iso"].apply(lambda x: x[:2])), on=["city", "iso"], how="left")
    .assign(locode=lambda d: d["locode"].mask(d["city"].eq("Washington, DC"), "USWAS"))
    .assign(locode=lambda d: d["locode"].mask(d["city"].eq("New York City"), "USNYC"))
    .assign(locode=lambda d: d["locode"].mask(d["city"].eq("São Paulo"), "BRSAO"))
    #.assign(locode=lambda d: d["locode"].mask(d["city"].eq("Dhaka North"), "BDDAN")) # need to add thesse to locode table
    #.assign(locode=lambda d: d["locode"].mask(d["city"].eq("Dhaka South"), "BDDAS"))
    ) #.query("locode.isna()"))

df_merged.head()

,city,name,iso,iso3,locode
0,Abidjan,Côte d'Ivoire,CI,CIV,CIABJ
1,Accra,Ghana,GH,GHA,GHACC
2,Addis Ababa,Ethiopia,ET,ETH,ETADD
3,Ahmedabad,India,IN,IND,INAMD
4,Amman,Jordan,JO,JOR,JOAMM


In [20]:
#this also has Population, Area, and GDP that may want to extract
df_out = (
    df_emissions
    .rename(columns={"City":"city"})
    .assign(city=lambda d: d["city"].replace(city_rename))
    .merge(df_merged[["city", "locode"]], on="city")
    .rename(columns={"locode":"actor_id", "Year_calendar":"year", "sector":"sector_id"})
    .loc[:,["actor_id", "year", "sector_id", "emissions"]]
)

In [21]:
#df_emissions["emissions"].astype(float)

In [22]:
df_out.head()

,actor_id,year,sector_id,emissions
0,CIABJ,2016,I.1.1,3.241273e+06
1,GHACC,2020,I.1.1,1.554908e+05
2,GHACC,2018,I.1.1,1.485428e+05
3,GHACC,2015,I.1.1,2.154585e+05
4,ETADD,2021,I.1.1,5.778799e+05


In [ ]:
#df_tmp = df_gpc.query("Scope == 1.0")

sector_dictionary = {
    "Stationary": "I",
    "Transport": "II",
    "Waste": "III",
    "IPUU": "IV",
    "AFOLU": "V"
}

sector_list = [
        {
            "code": "I",
            "name": "Stationary",
            "description": "Emissions from stationary energy production and use",
            "taxonomy": "GPC",
        },
        {
            "code": "II",
            "name": "Transport",
            "description": "Emissions from fuel combustion for transportation occurring in the city",
            "taxonomy": "GPC",
        },
        {
            "code":  "III",
            "name": "Waste",
            "description": "Emissions from in-boundary waste treatment",
            "taxonomy":  "GPC",
        },
        {
            "code": "IV",
            "name": "IPPU",
            "description": "Emissions from industrial processes and product use occurring within the city boundary",
            "taxonomy":  "GPC",
        },
        {
            "code": "V",
            "name": "AFOLU",
            "description": "Emissions from agricultural, other land-use and landuse-change",
            "taxonomy":  "GPC",
        },
    ]

df_sector = pd.DataFrame(sector_list)

df_subsector = (
    df_gpc
    .query("Scope == 1.0")
    .loc[:,["GPC ref", "Description", "Sub-Sector"]]
    .rename(columns = {"GPC ref": "code", "Sub-Sector":"name", "Description": "description"})
    .assign(
        taxonomy = "GPC",
        parent_code = lambda x: x["code"].apply(lambda row: row.split(".")[0]))
)

In [24]:
df_sector.head()

,code,name,description,taxonomy
0,I,Stationary,Emissions from stationary energy production an...,GPC
1,II,Transport,Emissions from fuel combustion for transportat...,GPC
2,III,Waste,Emissions from in-boundary waste treatment,GPC
3,IV,IPUU,Emissions from industrial processes and produc...,GPC
4,V,AFOLU,"Emissions from agricultural, other land-use an...",GPC


In [25]:
df_subsector = (
    df_gpc
    .query("Scope == 1.0")
    .loc[:,["GPC ref", "Description", "Sub-Sector"]]
    .rename(columns = {"GPC ref": "code", "Sub-Sector":"name", "Description": "description"})
    .assign(
        taxonomy = "GPC",
        parent_code = lambda x: x["code"].apply(lambda row: row.split(".")[0]))
)

In [29]:
df_subsector.head()

,code,description,name,taxonomy,parent_code
0,I.1.1,Emissions from fuel combustion within the city...,Residential Buildings,GPC,I
3,I.2.1,Emissions from fuel combustion within the city...,Commercial and institutional buildings and fac...,GPC,I
6,I.3.1,Emissions from fuel combustion within the city...,Manufacturing industries and construction,GPC,I
9,I.4.1,Emissions from energy used in power plant auxi...,Energy industries,GPC,I
12,I.4.4,Emissions from energy generation supplied to t...,Energy generation supplied to the grid,GPC,I


In [ ]:
#{sector}.{subsector}.{scope}

In [ ]:
id, code, parent_code, name, taxonomy, descroiption, datasource_id


In [ ]:
df_sector_data = pd.concat([df_sector, df_subsector])

,code,name,description,taxonomy,parent_code
0,I,Stationary,Emissions from stationary energy production an...,GPC,NaN
1,II,Transport,Emissions from fuel combustion for transportat...,GPC,NaN
2,III,Waste,Emissions from in-boundary waste treatment,GPC,NaN
3,IV,IPUU,Emissions from industrial processes and produc...,GPC,NaN
4,V,AFOLU,"Emissions from agricultural, other land-use an...",GPC,NaN
0,I.1.1,Residential Buildings,Emissions from fuel combustion within the city...,GPC,I
3,I.2.1,Commercial and institutional buildings and fac...,Emissions from fuel combustion within the city...,GPC,I
6,I.3.1,Manufacturing industries and construction,Emissions from fuel combustion within the city...,GPC,I
9,I.4.1,Energy industries,Emissions from energy used in power plant auxi...,GPC,I
12,I.4.4,Energy generation supplied to the grid,Emissions from energy generation supplied to t...,GPC,I


In [26]:
df_sector.concat(df_subsector)

AttributeError: 'DataFrame' object has no attribute 'concat'

In [28]:
df_out.head()

,actor_id,year,sector_id,emissions
0,CIABJ,2016,I.1.1,3.241273e+06
1,GHACC,2020,I.1.1,1.554908e+05
2,GHACC,2018,I.1.1,1.485428e+05
3,GHACC,2015,I.1.1,2.154585e+05
4,ETADD,2021,I.1.1,5.778799e+05


In [31]:

df_sector_data = (
    pd.concat([df_sector, df_subsector])
    .fillna("")
    .astype({"code": str, "parent_code": str, "name": str})
    )

In [ ]:
    # ------------------------------------------
    # emissions aggreate sector table
    # ------------------------------------------
    # this includes land
    df_emissions_total = (
        df_out
        .merge(df_sector_data[["code", "parent_code"]], left_on="sector_id", right_on="code")
        .groupby(["year", "actor_id", "parent_code"], as_index=False)
        .agg(emissions=("emissions", "sum"))
    )

    # this excludes land emissions
    df_emissions_total_ex = (
        df_out
        .query("sector_id != 'V.2'")
        .merge(df_sector_data[["code", "parent_code"]], left_on="sector_id", right_on="code")
        .groupby(["year", "actor_id", "parent_code"], as_index=False)
        .agg(emissions=("emissions", "sum"))
    )


,year,actor_id,parent_code,emissions
0,1990,CAMTR,I,8.609995e+06
1,1990,CAMTR,II,3.442388e+06
2,1990,CAMTR,III,1.708081e+06
3,1990,CAMTR,IV,6.095670e+05
4,1990,CAMTR,V,1.384000e+03
...,...,...,...,...
2485,2023,USMIO,II,1.096366e+06
2486,2023,USMIO,III,1.593520e+03
2487,2023,USMSY,I,6.074850e+05
2488,2023,USMSY,II,2.410426e+06


In [45]:
df_out

,actor_id,year,sector_id,emissions
0,CIABJ,2016,I.1.1,3.241273e+06
1,GHACC,2020,I.1.1,1.554908e+05
2,GHACC,2018,I.1.1,1.485428e+05
3,GHACC,2015,I.1.1,2.154585e+05
4,ETADD,2021,I.1.1,5.778799e+05
...,...,...,...,...
13333,AUSYD,2008,VI.1,7.546511e+05
13334,AUSYD,2007,VI.1,7.691602e+05
13335,AUSYD,2006,VI.1,7.760715e+05
13336,AUSYD,2005,VI.1,7.689420e+05


In [48]:
df_emissions_total = (
    df_out
    .groupby(["year", "actor_id"], as_index=False)
    .agg(emissions=("emissions", "sum"))
    .sort_values(by=["actor_id", "year"])
)

df_emissions_total

,year,actor_id,emissions
271,2016,AEDXB,6.714256e+07
460,2019,AEDXB,6.837452e+07
660,2022,AEDXB,5.577236e+07
3,2000,ARBUE,1.961151e+07
4,2001,ARBUE,1.570290e+07
...,...,...,...
216,2014,ZAJNB,2.199926e+07
334,2016,ZAJNB,2.395966e+07
217,2014,ZAPRY,2.153139e+07
270,2015,ZAPRY,2.051052e+07


In [52]:
#total emissions
# this includes land
df_total_inc = (
    df_out
    .groupby(["year", "actor_id"], as_index=False)
    .agg(emissions=("emissions", "sum"))
    .sort_values(by=["actor_id", "year"])
)

# this excludes land emissions
df_total_ex = (
    df_out
    .query("sector_id != 'V.2'")
    #.merge(df_sector_data[["code", "parent_code"]], left_on="sector_id", right_on="code")
    .groupby(["year", "actor_id"], as_index=False)
    .agg(emissions=("emissions", "sum"))
    .sort_values(by=["actor_id", "year"])
)

In [53]:
df_total_inc

,year,actor_id,emissions
271,2016,AEDXB,6.714256e+07
460,2019,AEDXB,6.837452e+07
660,2022,AEDXB,5.577236e+07
3,2000,ARBUE,1.961151e+07
4,2001,ARBUE,1.570290e+07
...,...,...,...
216,2014,ZAJNB,2.199926e+07
334,2016,ZAJNB,2.395966e+07
217,2014,ZAPRY,2.153139e+07
270,2015,ZAPRY,2.051052e+07


In [54]:
df_total_ex

,year,actor_id,emissions
271,2016,AEDXB,6.714256e+07
460,2019,AEDXB,6.837452e+07
660,2022,AEDXB,5.577236e+07
3,2000,ARBUE,1.961151e+07
4,2001,ARBUE,1.570290e+07
...,...,...,...
216,2014,ZAJNB,2.199926e+07
334,2016,ZAJNB,2.395966e+07
217,2014,ZAPRY,2.153139e+07
270,2015,ZAPRY,2.051052e+07
